# 01 — Raw Files Coverage Diagnostics

This notebook comes after:

`00_Make_Raw_Files_Comparable.ipynb`

Purpose:

Diagnose coverage of the standardized comparable country-year-variable dataset.

This notebook does:

- read the Step 00 combined long dataset;
- check variable-level coverage;
- check country-level coverage using the **core coverage variable set**;
- exclude raw public debt source variables from country coverage scoring;
- check country-variable coverage;
- check country-year coverage;
- check baseline-year coverage for 2007 and 2019;
- check pre-shock window coverage;
- check GDP growth availability for recovery construction;
- create diagnostic scorecards;
- export audit tables.

This notebook does **not**:

- choose the final country sample;
- keep only OECD countries;
- impute missing values;
- clip values;
- direction-align variables;
- create POSet-ready variables;
- process WGI.

WGI will be processed later in:

`03_WGI_Governance_Compilation.ipynb`

Final country and variable decisions happen later during master dataset construction and pre-POSet EDA.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "00_Comparable_Raw_Files"
    / "Combined_Long"
    / "all_comparable_sources_long.csv"
)

DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "01_Coverage_Diagnostics"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS and DIAGNOSTICS_DIR.exists():
    shutil.rmtree(DIAGNOSTICS_DIR)

DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Input file:", INPUT_FILE.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Audit folder:", AUDIT_DIR.resolve())

Run ID: 20260622_025639
Input file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\01_Coverage_Diagnostics
Audit folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit


In [2]:
# ------------------------------------------------------
# LOAD STEP 00 COMPARABLE LONG DATASET
# ------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE)

required_columns = {
    "Country",
    "Year",
    "variable",
    "Value",
    "source_file",
    "source_family",
    "concept",
    "expected_unit",
    "raw_direction",
    "is_derived",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Input dataset is missing required columns: {missing_columns}")

df["Country"] = df["Country"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

df = df.dropna(subset=["Country", "Year", "variable", "Value"]).copy()
df["Year"] = df["Year"].astype(int)

df = df.sort_values(["variable", "Country", "Year"]).reset_index(drop=True)

df.to_csv(
    DIAGNOSTICS_DIR / "source_data_snapshot.csv",
    index=False,
)

print("Rows:", len(df))
print("Countries:", df["Country"].nunique())
print("Variables:", df["variable"].nunique())
print("Years:", df["Year"].min(), "-", df["Year"].max())

display(df.head())

Rows: 13446
Countries: 169
Variables: 10
Years: 2000 - 2025


,Country,Country_Raw,country_name,Year,variable,Value,source_file,source_family,concept,expected_unit,raw_direction,is_derived,variable_notes,standardization_status,is_aggregate_region,Source_Used
0,AGO,AGO,Angola,2000,energy_import_dependency_raw,-546.6450,WorldBank_Energy_Import_Dependency_Raw_2000_20...,WorldBank,Energy import dependency,percent_of_energy_use_raw,higher_worse,False,Raw values preserved. Negative values are mean...,already_iso3,False,NaN
1,AGO,AGO,Angola,2001,energy_import_dependency_raw,-487.0143,WorldBank_Energy_Import_Dependency_Raw_2000_20...,WorldBank,Energy import dependency,percent_of_energy_use_raw,higher_worse,False,Raw values preserved. Negative values are mean...,already_iso3,False,NaN
2,AGO,AGO,Angola,2002,energy_import_dependency_raw,-575.4543,WorldBank_Energy_Import_Dependency_Raw_2000_20...,WorldBank,Energy import dependency,percent_of_energy_use_raw,higher_worse,False,Raw values preserved. Negative values are mean...,already_iso3,False,NaN
3,AGO,AGO,Angola,2003,energy_import_dependency_raw,-520.1810,WorldBank_Energy_Import_Dependency_Raw_2000_20...,WorldBank,Energy import dependency,percent_of_energy_use_raw,higher_worse,False,Raw values preserved. Negative values are mean...,already_iso3,False,NaN
4,AGO,AGO,Angola,2004,energy_import_dependency_raw,-590.3533,WorldBank_Energy_Import_Dependency_Raw_2000_20...,WorldBank,Energy import dependency,percent_of_energy_use_raw,higher_worse,False,Raw values preserved. Negative values are mean...,already_iso3,False,NaN


In [3]:
# ------------------------------------------------------
# CORE COVERAGE VARIABLE SET
# ------------------------------------------------------
# Raw Eurostat and World Bank public debt variables stay in the dataset for audit,
# but they are NOT used for country coverage scoring.
#
# This avoids counting the same public-debt concept three times.

SOURCE_DIAGNOSTIC_ONLY_VARIABLES = {
    "public_debt_gdp_eurostat",
    "public_debt_gdp_worldbank",
}

COVERAGE_CORE_VARIABLES = [
    "rd_gdp",
    "inflation_cpi",
    "unemployment_rate",
    "tertiary_education_25_64",
    "productivity_gdp_per_hour",
    "gdp_growth",
    "energy_import_dependency_raw",
    "public_debt_gdp_canonical",
]

missing_core_variables = sorted(set(COVERAGE_CORE_VARIABLES) - set(df["variable"].unique()))

if missing_core_variables:
    raise ValueError(f"Missing expected core coverage variables: {missing_core_variables}")

df_core = df[df["variable"].isin(COVERAGE_CORE_VARIABLES)].copy()

print("All variables in source data:", df["variable"].nunique())
print("Core variables used for coverage scoring:", df_core["variable"].nunique())
print("Source-diagnostic-only variables excluded from scoring:", sorted(SOURCE_DIAGNOSTIC_ONLY_VARIABLES))

display(pd.DataFrame({
    "coverage_core_variable": COVERAGE_CORE_VARIABLES
}))

All variables in source data: 10
Core variables used for coverage scoring: 8
Source-diagnostic-only variables excluded from scoring: ['public_debt_gdp_eurostat', 'public_debt_gdp_worldbank']


,coverage_core_variable
0,rd_gdp
1,inflation_cpi
2,unemployment_rate
3,tertiary_education_25_64
4,productivity_gdp_per_hour
5,gdp_growth
6,energy_import_dependency_raw
7,public_debt_gdp_canonical


In [4]:
# ------------------------------------------------------
# BASIC DUPLICATE CHECK
# ------------------------------------------------------
# This checks the full source dataset, including source-audit variables.

duplicate_check = (
    df
    .groupby(["Country", "Year", "variable"])
    .agg(
        row_count=("Value", "size"),
        unique_values=("Value", "nunique"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .query("row_count > 1")
    .sort_values(["row_count", "Country", "Year", "variable"], ascending=[False, True, True, True])
)

duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "duplicate_country_year_variable_check.csv",
    index=False,
)

display(duplicate_check.head(50))

if len(duplicate_check.query("unique_values > 1")) > 0:
    print("WARNING: Conflicting duplicate Country-Year-Variable rows exist.")
else:
    print("Duplicate check complete. No conflicting duplicates found.")

,Country,Year,variable,row_count,unique_values,min_value,max_value


Duplicate check complete. No conflicting duplicates found.


In [5]:
# ------------------------------------------------------
# VARIABLE-LEVEL COVERAGE SUMMARY
# ------------------------------------------------------
# This uses the full dataset so source-audit variables are visible.

all_countries = sorted(df["Country"].unique())
all_years = list(range(df["Year"].min(), df["Year"].max() + 1))

variable_coverage_summary = (
    df
    .groupby("variable")
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        years_available=("Year", "nunique"),
        min_value=("Value", "min"),
        median_value=("Value", "median"),
        max_value=("Value", "max"),
        source_families=("source_family", lambda x: ", ".join(sorted(set(x.astype(str))))),
        is_derived=("is_derived", "max"),
        concept=("concept", "first"),
        expected_unit=("expected_unit", "first"),
        raw_direction=("raw_direction", "first"),
    )
    .reset_index()
)

variable_coverage_summary["country_coverage_rate_vs_all_countries"] = (
    variable_coverage_summary["countries"] / len(all_countries)
)

variable_coverage_summary["year_coverage_rate_vs_global_year_span"] = (
    variable_coverage_summary["years_available"] / len(all_years)
)

variable_coverage_summary["coverage_role"] = np.where(
    variable_coverage_summary["variable"].isin(SOURCE_DIAGNOSTIC_ONLY_VARIABLES),
    "source_diagnostic_only",
    np.where(
        variable_coverage_summary["variable"].isin(COVERAGE_CORE_VARIABLES),
        "core_coverage_variable",
        "review"
    )
)

variable_coverage_summary = variable_coverage_summary.sort_values(
    ["coverage_role", "countries", "years_available"],
    ascending=[True, False, False],
).reset_index(drop=True)

variable_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "variable_coverage_summary.csv",
    index=False,
)

display(variable_coverage_summary)

,variable,rows,countries,min_year,max_year,years_available,min_value,median_value,max_value,source_families,is_derived,concept,expected_unit,raw_direction,country_coverage_rate_vs_all_countries,year_coverage_rate_vs_global_year_span,coverage_role
0,energy_import_dependency_raw,3151,145,2000,2023,24,-3300.1473,24.3911,450.1815,WorldBank,False,Energy import dependency,percent_of_energy_use_raw,higher_worse,0.8580,0.9231,core_coverage_variable
1,public_debt_gdp_canonical,1783,112,2000,2025,26,-1.1707,50.3000,209.4000,Derived,True,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,0.6627,1.0000,core_coverage_variable
2,unemployment_rate,1285,53,2000,2025,26,0.2510,6.5590,36.0250,OECD,False,Unemployment rate,percent_of_labour_force,higher_worse,0.3136,1.0000,core_coverage_variable
3,inflation_cpi,1210,48,2000,2025,26,-4.4475,2.6000,219.8845,OECD,False,Consumer price inflation,annual_percent_change,higher_generally_worse,0.2840,1.0000,core_coverage_variable
4,tertiary_education_25_64,973,48,2000,2024,25,5.4441,30.7121,64.6622,OECD,False,"Adult tertiary educational attainment, age 25-64",percent_of_same_age_population,higher_better,0.2840,0.9615,core_coverage_variable
5,rd_gdp,1077,47,2000,2025,26,0.1401,1.5436,6.9589,OECD,False,R&D expenditure as % of GDP,percent_of_gdp,higher_better,0.2781,1.0000,core_coverage_variable
6,gdp_growth,1131,44,2000,2025,26,-16.0400,2.6231,24.6240,OECD_DBnomics,False,Real GDP growth,annual_percent_change,context_dependent,0.2604,1.0000,core_coverage_variable
7,productivity_gdp_per_hour,1003,41,2000,2024,25,11.7448,51.2618,142.9044,OECD_DBnomics,False,GDP per hour worked,usd_ppp_per_hour_constant_prices,higher_better,0.2426,0.9615,core_coverage_variable
8,public_debt_gdp_worldbank,1131,87,2000,2024,25,-1.1707,49.9647,194.6828,WorldBank,False,Public debt as % of GDP,percent_of_gdp,higher_worse,0.5148,0.9615,source_diagnostic_only
9,public_debt_gdp_eurostat,702,27,2000,2025,26,3.9000,53.6500,209.4000,Eurostat,False,Public debt as % of GDP,percent_of_gdp,higher_worse,0.1598,1.0000,source_diagnostic_only


In [6]:
# ------------------------------------------------------
# VARIABLE RECOMMENDATION DIAGNOSTICS
# ------------------------------------------------------
# Diagnostic only.
# This does not finalize variable selection for POSet.

rows = []

for _, row in variable_coverage_summary.iterrows():
    variable = row["variable"]

    if variable in SOURCE_DIAGNOSTIC_ONLY_VARIABLES:
        status = "source_diagnostic_only"
        reason = "Raw public debt source kept for audit; canonical debt should be used for coverage/master dataset."
    elif variable in COVERAGE_CORE_VARIABLES:
        if row["countries"] >= 30 and row["years_available"] >= 15:
            status = "strong_master_candidate"
            reason = "Good country-year coverage and conceptually relevant."
        else:
            status = "review_master_candidate"
            reason = "Conceptually relevant but coverage should be reviewed."
    else:
        status = "review"
        reason = "Variable not explicitly classified."

    rows.append({
        "variable": variable,
        "coverage_status": status,
        "reason": reason,
        "coverage_role": row["coverage_role"],
        "rows": row["rows"],
        "countries": row["countries"],
        "years_available": row["years_available"],
        "min_year": row["min_year"],
        "max_year": row["max_year"],
        "concept": row["concept"],
    })

recommended_variable_keep_drop_review_list = pd.DataFrame(rows)

recommended_variable_keep_drop_review_list.to_csv(
    DIAGNOSTICS_DIR / "recommended_variable_keep_drop_review_list.csv",
    index=False,
)

display(recommended_variable_keep_drop_review_list)

,variable,coverage_status,reason,coverage_role,rows,countries,years_available,min_year,max_year,concept
0,energy_import_dependency_raw,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,3151,145,24,2000,2023,Energy import dependency
1,public_debt_gdp_canonical,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1783,112,26,2000,2025,Canonical public debt as % of GDP
2,unemployment_rate,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1285,53,26,2000,2025,Unemployment rate
3,inflation_cpi,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1210,48,26,2000,2025,Consumer price inflation
4,tertiary_education_25_64,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,973,48,25,2000,2024,"Adult tertiary educational attainment, age 25-64"
5,rd_gdp,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1077,47,26,2000,2025,R&D expenditure as % of GDP
6,gdp_growth,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1131,44,26,2000,2025,Real GDP growth
7,productivity_gdp_per_hour,strong_master_candidate,Good country-year coverage and conceptually re...,core_coverage_variable,1003,41,25,2000,2024,GDP per hour worked
8,public_debt_gdp_worldbank,source_diagnostic_only,Raw public debt source kept for audit; canonic...,source_diagnostic_only,1131,87,25,2000,2024,Public debt as % of GDP
9,public_debt_gdp_eurostat,source_diagnostic_only,Raw public debt source kept for audit; canonic...,source_diagnostic_only,702,27,26,2000,2025,Public debt as % of GDP


In [7]:
# ------------------------------------------------------
# COUNTRY-VARIABLE COVERAGE SUMMARY
# ------------------------------------------------------
# Uses core coverage variables only.

variables = COVERAGE_CORE_VARIABLES

country_variable_coverage_summary = (
    df_core
    .groupby(["Country", "variable"])
    .agg(
        rows=("Value", "size"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        years_available=("Year", "nunique"),
        min_value=("Value", "min"),
        median_value=("Value", "median"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values(["Country", "variable"])
)

country_variable_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "country_variable_coverage_summary.csv",
    index=False,
)

display(country_variable_coverage_summary.head(50))

,Country,variable,rows,min_year,max_year,years_available,min_value,median_value,max_value
0,AGO,energy_import_dependency_raw,23,2000,2022,23,-910.8359,-590.3533,-353.9718
1,ALB,energy_import_dependency_raw,24,2000,2023,24,12.6965,36.1824,54.0254
2,ALB,public_debt_gdp_canonical,14,2011,2024,14,49.9643,71.7053,83.4542
3,AND,public_debt_gdp_canonical,7,2018,2024,7,38.8380,42.5637,58.7561
4,ARE,energy_import_dependency_raw,23,2000,2022,23,-362.4749,-189.0374,-138.1301
5,ARE,public_debt_gdp_canonical,1,2013,2013,1,1.8033,1.8033,1.8033
6,ARG,energy_import_dependency_raw,24,2000,2023,24,-44.6310,0.8219,14.0678
7,ARG,inflation_cpi,8,2018,2025,8,34.2772,50.9788,219.8845
8,ARG,rd_gdp,24,2000,2023,24,0.3478,0.5269,0.6389
9,ARG,tertiary_education_25_64,17,2004,2023,17,16.7119,20.5561,24.6362


In [8]:
# ------------------------------------------------------
# COUNTRY-LEVEL MISSINGNESS SUMMARY
# ------------------------------------------------------
# Uses core coverage variables only.

country_variable_presence = (
    df_core
    .assign(present=1)
    .pivot_table(
        index="Country",
        columns="variable",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
)

for variable in variables:
    if variable not in country_variable_presence.columns:
        country_variable_presence[variable] = 0

country_variable_presence = country_variable_presence[variables]

country_variable_presence_matrix = country_variable_presence.reset_index()

country_variable_presence_matrix.to_csv(
    DIAGNOSTICS_DIR / "country_variable_presence_matrix.csv",
    index=False,
)

country_variable_year_count_matrix = (
    df_core
    .pivot_table(
        index="Country",
        columns="variable",
        values="Year",
        aggfunc="nunique",
        fill_value=0,
    )
    .reset_index()
)

for variable in variables:
    if variable not in country_variable_year_count_matrix.columns:
        country_variable_year_count_matrix[variable] = 0

country_variable_year_count_matrix = country_variable_year_count_matrix[["Country"] + variables]

country_variable_year_count_matrix.to_csv(
    DIAGNOSTICS_DIR / "country_variable_year_count_matrix.csv",
    index=False,
)

country_missingness_summary = country_variable_presence.copy()
country_missingness_summary["variables_present"] = country_missingness_summary.sum(axis=1)
country_missingness_summary["variables_missing"] = len(variables) - country_missingness_summary["variables_present"]
country_missingness_summary["variable_presence_rate"] = (
    country_missingness_summary["variables_present"] / len(variables)
)

country_missingness_summary = (
    country_missingness_summary
    .reset_index()
    [["Country", "variables_present", "variables_missing", "variable_presence_rate"] + variables]
    .sort_values(["variables_present", "Country"], ascending=[False, True])
    .reset_index(drop=True)
)

country_missingness_summary.to_csv(
    DIAGNOSTICS_DIR / "country_missingness_summary.csv",
    index=False,
)

display(country_missingness_summary.head(50))

variable,Country,variables_present,variables_missing,variable_presence_rate,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical
0,AUS,8,0,1.0000,1,1,1,1,1,1,1,1
1,AUT,8,0,1.0000,1,1,1,1,1,1,1,1
2,BEL,8,0,1.0000,1,1,1,1,1,1,1,1
3,CAN,8,0,1.0000,1,1,1,1,1,1,1,1
4,CHE,8,0,1.0000,1,1,1,1,1,1,1,1
5,CHL,8,0,1.0000,1,1,1,1,1,1,1,1
6,COL,8,0,1.0000,1,1,1,1,1,1,1,1
7,CRI,8,0,1.0000,1,1,1,1,1,1,1,1
8,CZE,8,0,1.0000,1,1,1,1,1,1,1,1
9,DEU,8,0,1.0000,1,1,1,1,1,1,1,1


In [9]:
# ------------------------------------------------------
# COUNTRY-YEAR COVERAGE SUMMARY
# ------------------------------------------------------
# Uses core coverage variables only.

country_year_coverage_summary = (
    df_core
    .groupby(["Country", "Year"])
    .agg(
        variables_available=("variable", "nunique"),
        rows=("Value", "size"),
    )
    .reset_index()
)

country_year_coverage_summary["variables_missing"] = (
    len(variables) - country_year_coverage_summary["variables_available"]
)

country_year_coverage_summary["coverage_rate"] = (
    country_year_coverage_summary["variables_available"] / len(variables)
)

country_year_coverage_summary = country_year_coverage_summary.sort_values(
    ["Country", "Year"]
).reset_index(drop=True)

country_year_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "country_year_coverage_summary.csv",
    index=False,
)

display(country_year_coverage_summary.head(50))

,Country,Year,variables_available,rows,variables_missing,coverage_rate
0,AGO,2000,1,1,7,0.1250
1,AGO,2001,1,1,7,0.1250
2,AGO,2002,1,1,7,0.1250
3,AGO,2003,1,1,7,0.1250
4,AGO,2004,1,1,7,0.1250
5,AGO,2005,1,1,7,0.1250
6,AGO,2006,1,1,7,0.1250
7,AGO,2007,1,1,7,0.1250
8,AGO,2008,1,1,7,0.1250
9,AGO,2009,1,1,7,0.1250


In [10]:
# ------------------------------------------------------
# YEAR-VARIABLE COVERAGE MATRIX
# ------------------------------------------------------
# Uses core coverage variables only.

year_variable_coverage_matrix = (
    df_core
    .pivot_table(
        index="Year",
        columns="variable",
        values="Country",
        aggfunc="nunique",
        fill_value=0,
    )
    .reset_index()
)

for variable in variables:
    if variable not in year_variable_coverage_matrix.columns:
        year_variable_coverage_matrix[variable] = 0

year_variable_coverage_matrix = year_variable_coverage_matrix[["Year"] + variables]

year_variable_coverage_matrix.to_csv(
    DIAGNOSTICS_DIR / "year_variable_coverage_matrix.csv",
    index=False,
)

display(year_variable_coverage_matrix)

variable,Year,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical
0,2000,38,47,45,31,40,44,129,66
1,2001,41,47,46,27,40,44,129,61
2,2002,39,47,47,29,40,44,129,64
3,2003,43,47,47,31,40,44,129,61
4,2004,43,47,47,33,40,44,131,58
5,2005,43,47,49,37,40,44,130,61
6,2006,43,47,49,36,40,44,130,66
7,2007,44,47,49,37,40,44,130,68
8,2008,45,47,50,38,40,44,131,75
9,2009,44,47,50,39,40,44,133,76


In [11]:
# ------------------------------------------------------
# BASELINE-YEAR COVERAGE CHECKS
# ------------------------------------------------------
# Uses core coverage variables only.
# 2007 and 2019 are key because they are used around the two recovery episodes.
# This still does not decide final sample.

BASELINE_YEARS = [2007, 2019]

baseline_rows = []

for baseline_year in BASELINE_YEARS:
    subset = df_core[df_core["Year"] == baseline_year].copy()

    presence = (
        subset
        .assign(present=1)
        .pivot_table(
            index="Country",
            columns="variable",
            values="present",
            aggfunc="max",
            fill_value=0,
        )
    )

    for variable in variables:
        if variable not in presence.columns:
            presence[variable] = 0

    presence = presence[variables]
    presence["baseline_year"] = baseline_year
    presence["variables_available"] = presence[variables].sum(axis=1)
    presence["variables_missing"] = len(variables) - presence["variables_available"]
    presence["coverage_rate"] = presence["variables_available"] / len(variables)

    baseline_rows.append(presence.reset_index())

baseline_year_coverage = pd.concat(baseline_rows, ignore_index=True)

baseline_year_coverage = baseline_year_coverage[
    ["Country", "baseline_year", "variables_available", "variables_missing", "coverage_rate"] + variables
].sort_values(["baseline_year", "variables_available", "Country"], ascending=[True, False, True])

baseline_year_coverage.to_csv(
    DIAGNOSTICS_DIR / "baseline_year_coverage_2007_2019.csv",
    index=False,
)

display(baseline_year_coverage.head(80))

variable,Country,baseline_year,variables_available,variables_missing,coverage_rate,rd_gdp,inflation_cpi,unemployment_rate,tertiary_education_25_64,productivity_gdp_per_hour,gdp_growth,energy_import_dependency_raw,public_debt_gdp_canonical
6,AUT,2007,8,0,1.0000,1,1,1,1,1,1,1,1
8,BEL,2007,8,0,1.0000,1,1,1,1,1,1,1,1
23,CAN,2007,8,0,1.0000,1,1,1,1,1,1,1,1
36,CZE,2007,8,0,1.0000,1,1,1,1,1,1,1,1
37,DEU,2007,8,0,1.0000,1,1,1,1,1,1,1,1
38,DNK,2007,8,0,1.0000,1,1,1,1,1,1,1,1
42,ESP,2007,8,0,1.0000,1,1,1,1,1,1,1,1
43,EST,2007,8,0,1.0000,1,1,1,1,1,1,1,1
44,FIN,2007,8,0,1.0000,1,1,1,1,1,1,1,1
45,FRA,2007,8,0,1.0000,1,1,1,1,1,1,1,1


In [12]:
# ------------------------------------------------------
# PRE-SHOCK WINDOW COVERAGE CHECKS
# ------------------------------------------------------
# Uses core coverage variables only.
#
# 2007 shock structural window: 2000–2007
# 2019 shock structural window: 2012–2019

PRE_SHOCK_WINDOWS = [
    {
        "shock_label": "shock_2007",
        "window_start": 2000,
        "window_end": 2007,
    },
    {
        "shock_label": "shock_2019",
        "window_start": 2012,
        "window_end": 2019,
    },
]

pre_shock_rows = []

for window in PRE_SHOCK_WINDOWS:
    shock_label = window["shock_label"]
    start = window["window_start"]
    end = window["window_end"]

    expected_years = list(range(start, end + 1))
    expected_year_count = len(expected_years)

    subset = df_core[
        (df_core["Year"] >= start)
        & (df_core["Year"] <= end)
    ].copy()

    summary = (
        subset
        .groupby(["Country", "variable"])
        .agg(
            years_available=("Year", "nunique"),
            min_year=("Year", "min"),
            max_year=("Year", "max"),
        )
        .reset_index()
    )

    summary["shock_label"] = shock_label
    summary["window_start"] = start
    summary["window_end"] = end
    summary["expected_year_count"] = expected_year_count
    summary["coverage_rate_in_window"] = summary["years_available"] / expected_year_count

    pre_shock_rows.append(summary)

pre_shock_window_coverage = pd.concat(pre_shock_rows, ignore_index=True)

pre_shock_window_coverage = pre_shock_window_coverage[
    [
        "shock_label",
        "Country",
        "variable",
        "window_start",
        "window_end",
        "expected_year_count",
        "years_available",
        "coverage_rate_in_window",
        "min_year",
        "max_year",
    ]
].sort_values(["shock_label", "Country", "variable"])

pre_shock_window_coverage.to_csv(
    DIAGNOSTICS_DIR / "pre_shock_window_coverage.csv",
    index=False,
)

display(pre_shock_window_coverage.head(80))

,shock_label,Country,variable,window_start,window_end,expected_year_count,years_available,coverage_rate_in_window,min_year,max_year
0,shock_2007,AGO,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
1,shock_2007,ALB,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
2,shock_2007,ARE,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
3,shock_2007,ARG,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
4,shock_2007,ARG,rd_gdp,2000,2007,8,8,1.0000,2000,2007
5,shock_2007,ARG,tertiary_education_25_64,2000,2007,8,3,0.3750,2004,2006
6,shock_2007,ARM,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
7,shock_2007,AUS,energy_import_dependency_raw,2000,2007,8,8,1.0000,2000,2007
8,shock_2007,AUS,gdp_growth,2000,2007,8,8,1.0000,2000,2007
9,shock_2007,AUS,inflation_cpi,2000,2007,8,8,1.0000,2000,2007


In [13]:
# ------------------------------------------------------
# GDP GROWTH RECOVERY INPUT COVERAGE
# ------------------------------------------------------
# This checks whether GDP growth exists around the recovery windows.
# It does not construct recovery outcomes yet.

GDP_RECOVERY_CHECK_WINDOWS = [
    {
        "recovery_case": "recovery_2007",
        "required_start_year": 2007,
        "required_end_year": 2025,
        "shock_window_years": [2008, 2009],
    },
    {
        "recovery_case": "recovery_2019",
        "required_start_year": 2019,
        "required_end_year": 2025,
        "shock_window_years": [2020, 2021],
    },
]

gdp_df = df_core[df_core["variable"] == "gdp_growth"].copy()

gdp_rows = []

for case in GDP_RECOVERY_CHECK_WINDOWS:
    case_label = case["recovery_case"]
    start = case["required_start_year"]
    end = case["required_end_year"]
    required_years = list(range(start, end + 1))
    shock_years = case["shock_window_years"]

    subset = gdp_df[
        (gdp_df["Year"] >= start)
        & (gdp_df["Year"] <= end)
    ].copy()

    for country in sorted(gdp_df["Country"].unique()):
        country_subset = subset[subset["Country"] == country].copy()
        years_available = sorted(country_subset["Year"].unique().tolist())

        shock_window_available = all(y in years_available for y in shock_years)
        baseline_available = start in years_available

        gdp_rows.append({
            "recovery_case": case_label,
            "Country": country,
            "required_start_year": start,
            "required_end_year": end,
            "expected_year_count": len(required_years),
            "years_available": len(years_available),
            "coverage_rate": len(years_available) / len(required_years),
            "baseline_available": baseline_available,
            "shock_window_years": ",".join(map(str, shock_years)),
            "shock_window_available": shock_window_available,
            "min_available_year": min(years_available) if years_available else np.nan,
            "max_available_year": max(years_available) if years_available else np.nan,
            "years_available_list": ",".join(map(str, years_available)),
        })

gdp_growth_recovery_input_coverage = pd.DataFrame(gdp_rows)

gdp_growth_recovery_input_coverage.to_csv(
    DIAGNOSTICS_DIR / "gdp_growth_recovery_input_coverage.csv",
    index=False,
)

display(gdp_growth_recovery_input_coverage.head(80))

,recovery_case,Country,required_start_year,required_end_year,expected_year_count,years_available,coverage_rate,baseline_available,shock_window_years,shock_window_available,min_available_year,max_available_year,years_available_list
0,recovery_2007,AUS,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
1,recovery_2007,AUT,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
2,recovery_2007,BEL,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
3,recovery_2007,BRA,2007,2025,19,17,0.8947,True,"2008,2009",True,2007,2023,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
4,recovery_2007,CAN,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
5,recovery_2007,CHE,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
6,recovery_2007,CHL,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
7,recovery_2007,CHN,2007,2025,19,17,0.8947,True,"2008,2009",True,2007,2023,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
8,recovery_2007,COL,2007,2025,19,19,1.0000,True,"2008,2009",True,2007,2025,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."
9,recovery_2007,CRI,2007,2025,19,18,0.9474,True,"2008,2009",True,2007,2024,"2007,2008,2009,2010,2011,2012,2013,2014,2015,2..."


In [14]:
# ------------------------------------------------------
# COUNTRY QUALITY SCORECARD
# ------------------------------------------------------
# Uses core coverage variables only.
# This scorecard is diagnostic only.
# It helps identify strong/review/weak coverage countries.
# It does not finalize the sample.

variable_year_span = (
    df_core
    .groupby("variable")
    .agg(
        global_min_year=("Year", "min"),
        global_max_year=("Year", "max"),
        global_years_available=("Year", "nunique"),
    )
    .reset_index()
)

country_variable_years = (
    df_core
    .groupby(["Country", "variable"])
    .agg(country_years_available=("Year", "nunique"))
    .reset_index()
)

coverage_scoring = country_variable_years.merge(
    variable_year_span,
    on="variable",
    how="left",
)

coverage_scoring["coverage_rate_vs_variable_availability"] = (
    coverage_scoring["country_years_available"]
    / coverage_scoring["global_years_available"]
)

country_score = (
    coverage_scoring
    .groupby("Country")
    .agg(
        variables_present=("variable", "nunique"),
        mean_variable_coverage_rate=("coverage_rate_vs_variable_availability", "mean"),
        min_variable_coverage_rate=("coverage_rate_vs_variable_availability", "min"),
        total_country_variable_years=("country_years_available", "sum"),
    )
    .reset_index()
)

country_score["variables_missing"] = len(variables) - country_score["variables_present"]
country_score["variable_presence_rate"] = country_score["variables_present"] / len(variables)

baseline_country_score = (
    baseline_year_coverage
    .groupby("Country")
    .agg(
        baseline_years_checked=("baseline_year", "nunique"),
        mean_baseline_coverage_rate=("coverage_rate", "mean"),
        min_baseline_coverage_rate=("coverage_rate", "min"),
    )
    .reset_index()
)

country_score = country_score.merge(
    baseline_country_score,
    on="Country",
    how="left",
)

gdp_recovery_flags = (
    gdp_growth_recovery_input_coverage
    .pivot_table(
        index="Country",
        columns="recovery_case",
        values="shock_window_available",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)

country_score = country_score.merge(
    gdp_recovery_flags,
    on="Country",
    how="left",
)

for col in ["recovery_2007", "recovery_2019"]:
    if col not in country_score.columns:
        country_score[col] = False

country_score["has_both_gdp_recovery_shock_windows"] = (
    country_score["recovery_2007"].fillna(False).astype(bool)
    & country_score["recovery_2019"].fillna(False).astype(bool)
)

def classify_country_coverage(row):
    if (
        row["variables_present"] >= 7
        and row["mean_variable_coverage_rate"] >= 0.70
        and row["has_both_gdp_recovery_shock_windows"]
    ):
        return "strong_coverage_candidate"

    if (
        row["variables_present"] >= 5
        and row["mean_variable_coverage_rate"] >= 0.50
    ):
        return "review_candidate"

    return "weak_coverage_review"

country_score["coverage_recommendation"] = country_score.apply(classify_country_coverage, axis=1)

country_score["note"] = (
    "Diagnostic only; not final country selection. "
    "Scoring uses core coverage variables only and excludes raw public-debt source variables."
)

final_country_quality_scorecard = country_score.sort_values(
    [
        "coverage_recommendation",
        "variables_present",
        "mean_variable_coverage_rate",
        "Country",
    ],
    ascending=[True, False, False, True],
).reset_index(drop=True)

final_country_quality_scorecard.to_csv(
    DIAGNOSTICS_DIR / "final_country_quality_scorecard.csv",
    index=False,
)

recommended_country_keep_drop_review_list = final_country_quality_scorecard[
    [
        "Country",
        "coverage_recommendation",
        "variables_present",
        "variables_missing",
        "variable_presence_rate",
        "mean_variable_coverage_rate",
        "min_variable_coverage_rate",
        "mean_baseline_coverage_rate",
        "min_baseline_coverage_rate",
        "has_both_gdp_recovery_shock_windows",
        "note",
    ]
].copy()

recommended_country_keep_drop_review_list.to_csv(
    DIAGNOSTICS_DIR / "recommended_country_keep_drop_review_list.csv",
    index=False,
)

display(recommended_country_keep_drop_review_list.head(80))

,Country,coverage_recommendation,variables_present,variables_missing,variable_presence_rate,mean_variable_coverage_rate,min_variable_coverage_rate,mean_baseline_coverage_rate,min_baseline_coverage_rate,has_both_gdp_recovery_shock_windows,note
0,BGR,review_candidate,7,1,0.8750,0.8859,0.2400,0.8125,0.7500,False,Diagnostic only; not final country selection. ...
1,HRV,review_candidate,7,1,0.8750,0.8525,0.1600,0.8125,0.7500,False,Diagnostic only; not final country selection. ...
2,ROU,review_candidate,6,2,0.7500,0.8669,0.2400,0.6875,0.6250,False,Diagnostic only; not final country selection. ...
3,RUS,review_candidate,6,2,0.7500,0.8584,0.7692,0.7500,0.7500,False,Diagnostic only; not final country selection. ...
4,BRA,review_candidate,6,2,0.7500,0.8374,0.5769,0.6875,0.6250,True,Diagnostic only; not final country selection. ...
5,IDN,review_candidate,6,2,0.7500,0.7741,0.1538,0.6250,0.6250,True,Diagnostic only; not final country selection. ...
6,IND,review_candidate,6,2,0.7500,0.7521,0.3200,0.5625,0.5000,True,Diagnostic only; not final country selection. ...
7,CHN,review_candidate,6,2,0.7500,0.6736,0.0769,0.5000,0.5000,True,Diagnostic only; not final country selection. ...
8,ARG,review_candidate,5,3,0.6250,0.6437,0.3077,0.4375,0.2500,False,Diagnostic only; not final country selection. ...
9,BEL,strong_coverage_candidate,8,0,1.0000,0.9952,0.9615,1.0000,1.0000,True,Diagnostic only; not final country selection. ...


In [15]:
# ------------------------------------------------------
# COVERAGE SUMMARY TABLES
# ------------------------------------------------------

coverage_summary_overall = pd.DataFrame([{
    "run_id": RUN_ID,
    "timestamp": RUN_TIMESTAMP,
    "source_rows_all_variables": len(df),
    "source_countries_all_variables": df["Country"].nunique(),
    "source_variables_all": df["variable"].nunique(),
    "core_rows": len(df_core),
    "core_countries": df_core["Country"].nunique(),
    "core_variables_used_for_scoring": len(COVERAGE_CORE_VARIABLES),
    "source_diagnostic_only_variables": ",".join(sorted(SOURCE_DIAGNOSTIC_ONLY_VARIABLES)),
    "min_year": df["Year"].min(),
    "max_year": df["Year"].max(),
    "strong_coverage_candidates": int((recommended_country_keep_drop_review_list["coverage_recommendation"] == "strong_coverage_candidate").sum()),
    "review_candidates": int((recommended_country_keep_drop_review_list["coverage_recommendation"] == "review_candidate").sum()),
    "weak_coverage_review": int((recommended_country_keep_drop_review_list["coverage_recommendation"] == "weak_coverage_review").sum()),
}])

coverage_summary_overall.to_csv(
    DIAGNOSTICS_DIR / "coverage_summary_overall.csv",
    index=False,
)

coverage_recommendation_counts = (
    recommended_country_keep_drop_review_list
    .groupby("coverage_recommendation")
    .size()
    .reset_index(name="country_count")
    .sort_values("country_count", ascending=False)
)

coverage_recommendation_counts.to_csv(
    DIAGNOSTICS_DIR / "coverage_recommendation_counts.csv",
    index=False,
)

display(coverage_summary_overall)
display(coverage_recommendation_counts)

,run_id,timestamp,source_rows_all_variables,source_countries_all_variables,source_variables_all,core_rows,core_countries,core_variables_used_for_scoring,source_diagnostic_only_variables,min_year,max_year,strong_coverage_candidates,review_candidates,weak_coverage_review
0,20260622_025639,2026-06-22 02:56:39,13446,169,10,11613,169,8,"public_debt_gdp_eurostat,public_debt_gdp_world...",2000,2025,39,9,121


,coverage_recommendation,country_count
2,weak_coverage_review,121
1,strong_coverage_candidate,39
0,review_candidate,9


In [16]:
# ------------------------------------------------------
# CREATE COVERAGE DIAGNOSTICS AUDIT WORKBOOK
# ------------------------------------------------------

COVERAGE_AUDIT_XLSX = AUDIT_DIR / "01_Coverage_Diagnostics_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_overall_summary",
        "description": "Overall coverage summary.",
        "path": DIAGNOSTICS_DIR / "coverage_summary_overall.csv",
    },
    {
        "sheet_name": "02_variable_coverage",
        "description": "Coverage by variable. Includes source-audit variables.",
        "path": DIAGNOSTICS_DIR / "variable_coverage_summary.csv",
    },
    {
        "sheet_name": "03_variable_review",
        "description": "Diagnostic variable keep/review/source-only classification.",
        "path": DIAGNOSTICS_DIR / "recommended_variable_keep_drop_review_list.csv",
    },
    {
        "sheet_name": "04_country_missing",
        "description": "Country-level missingness summary using core variables only.",
        "path": DIAGNOSTICS_DIR / "country_missingness_summary.csv",
    },
    {
        "sheet_name": "05_country_quality",
        "description": "Country quality scorecard using core variables only.",
        "path": DIAGNOSTICS_DIR / "final_country_quality_scorecard.csv",
    },
    {
        "sheet_name": "06_country_review",
        "description": "Country coverage recommendation list using core variables only.",
        "path": DIAGNOSTICS_DIR / "recommended_country_keep_drop_review_list.csv",
    },
    {
        "sheet_name": "07_country_variable",
        "description": "Country-variable coverage summary using core variables only.",
        "path": DIAGNOSTICS_DIR / "country_variable_coverage_summary.csv",
    },
    {
        "sheet_name": "08_country_year",
        "description": "Country-year coverage summary using core variables only.",
        "path": DIAGNOSTICS_DIR / "country_year_coverage_summary.csv",
    },
    {
        "sheet_name": "09_year_variable",
        "description": "Year-variable country-count matrix using core variables only.",
        "path": DIAGNOSTICS_DIR / "year_variable_coverage_matrix.csv",
    },
    {
        "sheet_name": "10_baseline_years",
        "description": "Baseline year coverage for 2007 and 2019 using core variables only.",
        "path": DIAGNOSTICS_DIR / "baseline_year_coverage_2007_2019.csv",
    },
    {
        "sheet_name": "11_pre_shock",
        "description": "Pre-shock window coverage using core variables only.",
        "path": DIAGNOSTICS_DIR / "pre_shock_window_coverage.csv",
    },
    {
        "sheet_name": "12_gdp_recovery",
        "description": "GDP growth recovery input coverage.",
        "path": DIAGNOSTICS_DIR / "gdp_growth_recovery_input_coverage.csv",
    },
    {
        "sheet_name": "13_duplicates",
        "description": "Duplicate Country-Year-Variable check on full source dataset.",
        "path": DIAGNOSTICS_DIR / "duplicate_country_year_variable_check.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(COVERAGE_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "01 Coverage Diagnostics Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for country-variable-year coverage diagnostics.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Coverage audit workbook created:")
print(COVERAGE_AUDIT_XLSX.resolve())

Coverage audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\01_Coverage_Diagnostics_Audit.xlsx


In [17]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("01 RAW FILES COVERAGE DIAGNOSTICS COMPLETE")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(COVERAGE_AUDIT_XLSX.resolve())

print("\nMain outputs:")
main_outputs = [
    "source_data_snapshot.csv",
    "variable_coverage_summary.csv",
    "recommended_variable_keep_drop_review_list.csv",
    "country_missingness_summary.csv",
    "country_variable_coverage_summary.csv",
    "country_year_coverage_summary.csv",
    "country_variable_presence_matrix.csv",
    "country_variable_year_count_matrix.csv",
    "year_variable_coverage_matrix.csv",
    "baseline_year_coverage_2007_2019.csv",
    "pre_shock_window_coverage.csv",
    "gdp_growth_recovery_input_coverage.csv",
    "final_country_quality_scorecard.csv",
    "recommended_country_keep_drop_review_list.csv",
    "coverage_summary_overall.csv",
]

for file_name in main_outputs:
    path = DIAGNOSTICS_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nImportant notes:")
print("1. No final country/sample selection was done here.")
print("2. WGI was not processed here.")
print("3. Raw Eurostat and World Bank debt variables were kept visible but excluded from country coverage scoring.")
print("4. Canonical public debt is used as the coverage/master debt variable.")
print("5. GDP growth coverage was checked for later recovery construction.")
print("6. Baseline years 2007 and 2019 were checked for structural coverage.")
print("7. No imputation, clipping, direction alignment, or POSet transformation was done.")

print("\nNext notebook:")
print("02_GDP_Recovery_Dynamic_Baseline.ipynb")

01 RAW FILES COVERAGE DIAGNOSTICS COMPLETE

Input file:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\01_Coverage_Diagnostics

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\01_Coverage_Diagnostics_Audit.xlsx

Main outputs:
- FOUND: source_data_snapshot.csv
- FOUND: variable_coverage_summary.csv
- FOUND: recommended_variable_keep_drop_review_list.csv
- FOUND: country_missingness_summary.csv
- FOUND: country_variable_coverage_summary.csv
- FOUND: country_year_coverage_summary.csv
- FOUND: country_variable_presence_matrix.csv
- FOUND: country_variable_year_count_matrix.csv
- FOUND: year_variable_cov